In [ ]:
import pm4py

path = "data/example_3_ocel.json"
# path = "data/scenario_test_ocel.json"
ocel = pm4py.read_ocel2_json(path)

df_events = ocel.events.copy()
df_events.set_index("ocel:eid", inplace=True)
df_relations = ocel.relations.copy()
df_relations.set_index("ocel:eid", inplace=True)

df_events_objects = df_events.join(df_relations, rsuffix="_relations")

In [ ]:
import networkx as nx

from utils import load_graphml_with_json_attrs

graphml_path = path.replace(".json", ".graphml")
ocel_nx = load_graphml_with_json_attrs(graphml_path)

In [ ]:
from collections import Counter

from process_execution import extract_process_execution, ProcessExecution

object_types = ["PackingUnit"]

events_to_trace = df_events_objects[
    (df_events_objects["ocel:type"].isin(object_types))
].index.values

print(f"Number of events selected: {len(events_to_trace)}")


def determine_class_quality(G: nx.Graph, event: str):
    return G.nodes()[event]["attr"].get("averageQuality") >= 1.0


def determine_class_attribute(G: nx.Graph):
    selected_activity = "Object-departing-WB"
    selected_attribute = "a"
    for _, data in G.nodes(data="attr"):
        if (
            data.get("ocel:activity", "") == selected_activity
            and data.get(selected_attribute, 1) < 0.25
        ):
            return False
    return True


trace_graphs = {}
for event in events_to_trace:
    trace_graph = extract_process_execution(
        ocel_nx,
        event,
        ["ProductionLot", "PackingUnit"],
        "Object-creating_class_instance",
    )
    trace_graph.construct_node_label()
    trace_graph.construct_edge_label()

    trace_graphs[event] = {
        "process_execution": trace_graph,
        "class": determine_class_quality(ocel_nx, event),
        # "class": determine_class_attribute(trace_graph),
    }


Counter([d["class"] for d in trace_graphs.values()])

### Optimization (branch & bound)

In [ ]:
import json
import torch

from gnn_graph_classification import (
    convert_trace_graphs_to_pyg,
    GCNWithEdgeAgg,
)

model_path = "data/scenario_test_ocel-model.pth"

# Define vocabulary and numeric attribute keys
with open(model_path.replace("-model.pth", "-vocab.json")) as f:
    vocab_dict = json.load(f)
node_labels = vocab_dict["node_labels"]
node_numeric_keys = vocab_dict["node_numeric_keys"]
edge_numeric_keys = vocab_dict["edge_numeric_keys"]

# Load trained model
device = "cuda" if torch.cuda.is_available() else "cpu"
try:
    model
except NameError:
    model = torch.load(model_path, weights_only=False)

model = model.to(device)
model.eval()


def process_outcome(p: ProcessExecution):
    """Predict the outcome for a single `ProcessExecution` using the loaded GNN model.

    Args:
        p (ProcessExecution): The process execution to classify.
    Returns:
        bool: The predicted class label (True/False).
    """

    # Convert the single ProcessExecution into the converter's expected input
    try:
        graph_map = {"_tmp": {"process_execution": p}}
        data_list = convert_trace_graphs_to_pyg(
            graph_map, node_labels, node_numeric_keys, edge_numeric_keys
        )
        if not data_list:
            raise RuntimeError("converter returned empty list")
        data = data_list[0]
    except Exception as e:
        print("process_outcome: conversion to PyG data failed:", e)
        raise

    # Prepare tensors and batch vector for a single-graph forward pass
    try:
        import torch

        # create a batch vector of zeros (single graph)
        batch_vec = torch.zeros(data.x.size(0), dtype=torch.long, device=device)
        data = data.to(device)
    except Exception as e:
        print("process_outcome: torch/device preparation failed:", e)
        raise

    # Forward through the model
    try:
        model.eval()
        with torch.no_grad():
            out = model(data.x, data.edge_index, batch_vec)
            # probs = torch.nn.functional.softmax(out, dim=-1)

            # Expecting graph-level logits shaped (1, n_classes)
            if out.dim() == 2 and out.size(0) == 1:
                pred = int(out.argmax(dim=1).item())
            else:
                # Fallback: take global argmax
                pred = int(out.argmax().item())
    except Exception as e:
        print("process_outcome: model forward failed:", e)
        raise

    return bool(pred)

In [ ]:
def process_outcome(p: ProcessExecution):
    return "DB2" not in p.nodes()

In [ ]:
import logging

# Clear log file
with open("logs/debug.log", "w") as f:
    f.write("")

logging.basicConfig(filename="logs/debug.log", level=logging.DEBUG, force=True)
logger = logging.getLogger()

In [ ]:
import branch_and_bound

from importlib import reload

reload(branch_and_bound)

TBD: separate 'feature' for each object node and thus branch on each object node substitution, or combine all into one feature.

In [ ]:
from itertools import combinations, product
from numpy import arange

from analysis.branch_and_bound import (
    BranchAndBoundCounterFactual,
    Action,
    NodeAttributeNumeric,
    ObjectSubstitutions,
)

MAX_CHANGES = 1000
COUNTERFACTUAL_LABEL = True

selected_event_attributes = {
    "a": arange(0, 1, 0.1),
    "quantity": range(0, 1000, 50),
}

target_process_execution_id = "198"
# target_process_execution_id = "100023"
target_process_execution = trace_graphs[target_process_execution_id][
    "process_execution"
]

object_substitutions = []
allowed_substitutions = []
for node_id, data in target_process_execution.nodes(data=True):
    if data["attr"].get("type", "") != "OBJECT":
        continue

    substitutions = [
        ((node_id, data), (subst_id, subst_data))
        for subst_id, subst_data in ocel_nx.nodes(data=True)
        if subst_id != node_id
        and subst_data["attr"].get("ocel:type", "") == data["attr"].get("ocel:type", "")
        and subst_data["attr"].get("capability", "")
        == data["attr"].get("capability", "")
    ]

    allowed_substitutions.append(substitutions)

    object_substitutions.append(
        ObjectSubstitutions(
            substitution_options=[[subst] for subst in [()] + substitutions],
        )
    )

# allowed_substitutions += [[()]]
# object_substitutions = [
#     ObjectSubstitutions(
#         substitution_options=[
#             p
#             for r in range(len(allowed_substitutions) + 1)
#             for c in combinations(allowed_substitutions, r)
#             for p in product(*c)
#         ],
#     )
# ]

# Define event node attributes
event_node_attributes = [
    NodeAttributeNumeric(
        node_id=node_id,
        attribute_name=attr_name,
        value_range=selected_event_attributes[attr_name],
    )
    for node_id, attr in target_process_execution.nodes(data="attr")
    if attr.get("type", "") == "EVENT"
    for attr_name in attr.keys()
    if attr_name in selected_event_attributes
]

print(
    f"object_substitutions: {len(object_substitutions)}\nevent_node_attributes: {len(event_node_attributes)}"
)

branch_and_bound = BranchAndBoundCounterFactual(
    process_outcome=process_outcome,
    max_changes=MAX_CHANGES,
    counterfactual_label=COUNTERFACTUAL_LABEL,
    num_workers=20,
)

action = Action()
available_features = object_substitutions + event_node_attributes
fixed_features = []
selected_actions = branch_and_bound.find_counterfactuals(
    target_process_execution,
    available_features,
)

print(len(selected_actions))

print(selected_actions[0])

In [ ]:
for selected_action in selected_actions:
    print(
    [
        f"{feature}: {change_value}"
        for feature, change_value in selected_action.node_attributes_modification.items()
        if change_value != 0
    ],
    [
        (subst[0][0], subst[1][0])
        for v in selected_action.object_substitution.values()
        for subst in v
        if subst
    ],
)

### Instance-based

Two step approach:
1) Find *k* graphs with different class, but similar structure (including node labels);
2) Among the *k* graphs, find the most similar graph, also considering the node attributes.

In [ ]:
from grakel.kernels import (
    VertexHistogram,
    WeisfeilerLehman,
)
from grakel.utils import graph_from_networkx

import numpy as np

k = 10  # select top k structurally most similar graphs

target_trace_graph_id = "100023"
target_trace_graph = trace_graphs[target_trace_graph_id]["process_execution"]

selected_trace_graphs = {
    k: trace_graphs[k]["process_execution"] for k in (list(trace_graphs.keys()))
}

gk = WeisfeilerLehman(n_iter=2, normalize=True, base_graph_kernel=VertexHistogram)

target_trace_graph_grakel = graph_from_networkx(
    [target_trace_graph],
    node_labels_tag="label",
    as_Graph=True,
    # val_node_labels="test",
)
gk.fit(target_trace_graph_grakel)

selected_trace_graphs_grakel = graph_from_networkx(
    selected_trace_graphs.values(),
    node_labels_tag="label",
    as_Graph=True,
    # val_node_labels="test",
)
K_gk = gk.transform(selected_trace_graphs_grakel)

most_similar_trace_graphs_gk = np.array(list(selected_trace_graphs.keys()))[
    np.argsort(K_gk[:, 0])[(-1 * k) :]
]
print(most_similar_trace_graphs_gk)

In [ ]:
import numpy as np

from grakel.kernels import SubgraphMatching

selected_trace_graphs = {
    k: nx.DiGraph(trace_graphs[k]["process_execution"])
    for k in most_similar_trace_graphs_gk
    if trace_graphs[k]["class"] != trace_graphs[target_trace_graph_id]["class"]
}


def numeric_diff(a, b):
    try:
        return 1 - (float(b) - float(a)) / float(a)
    except ZeroDivisionError:
        return 0.5
    except ValueError:
        return 0.5


def dict_compare(a, b):
    sim_score = 0
    for key in a.keys():
        v_a = a.get(key)
        v_b = b.get(key)

        if not (v_a and v_b):
            sim_score += 0

        try:
            sim_score += 1 - (float(v_b) - float(v_a)) / float(v_a)
        except ZeroDivisionError:
            sim_score += 0.5
        except TypeError:
            sim_score += int(v_a == v_b)
        except ValueError:
            sim_score += int(v_a == v_b)
    return sim_score


sub_match = SubgraphMatching(
    normalize=True,
    kv=dict_compare,
    ke=None,
)

target_trace_graph_grakel = graph_from_networkx(
    [target_trace_graph],
    node_labels_tag="attr",
    # as_Graph=True,
    # val_node_labels="test",
    # edge_labels_tag="attr",
)
sub_match.fit(target_trace_graph_grakel)

selected_trace_graphs_grakel = graph_from_networkx(
    selected_trace_graphs.values(),
    node_labels_tag="attr",
    # as_Graph=True,
    # val_node_labels="test",
    edge_labels_tag="attr",
)
K = sub_match.transform(selected_trace_graphs_grakel)

most_similar_trace_graph_id = list(selected_trace_graphs.keys())[
    np.argsort(K[:, 0])[-1]
]

print("Query process execution")
print("--------------")
print(target_trace_graph_id)
print()
print("Most similar process execution")
print("---------------------")
print(most_similar_trace_graph_id)

### Instance-based + optimization

Two step approach:
1) Find *k* graphs with different class, but similar structure (including node labels);
2) From the *k* similar graphs and optimize objective function by modifying node attributes.

In [ ]:
from grakel.kernels import (
    VertexHistogram,
    WeisfeilerLehman,
)
from grakel.utils import graph_from_networkx

import numpy as np

k = 10  # select top k structurally most similar graphs

target_trace_graph_id = "100023"
target_trace_graph = trace_graphs[target_trace_graph_id]["process_execution"]

selected_trace_graphs = {
    k: trace_graphs[k]["process_execution"]
    for k in (list(trace_graphs.keys()))
    if trace_graphs[k]["class"] != trace_graphs[target_trace_graph_id]["class"]
}

gk = WeisfeilerLehman(n_iter=2, normalize=True, base_graph_kernel=VertexHistogram)

target_trace_graph_grakel = graph_from_networkx(
    [target_trace_graph],
    node_labels_tag="label",
    as_Graph=True,
    # val_node_labels="test",
)
gk.fit(target_trace_graph_grakel)

selected_trace_graphs_grakel = graph_from_networkx(
    selected_trace_graphs.values(),
    node_labels_tag="label",
    as_Graph=True,
    # val_node_labels="test",
)
K_gk = gk.transform(selected_trace_graphs_grakel)

most_similar_trace_graphs_gk = np.array(list(selected_trace_graphs.keys()))[
    np.argsort(K_gk[:, 0])[(-1 * k) :]
]
print(most_similar_trace_graphs_gk)

In [ ]:
from gnn_graph_classification import (
    build_vocab_and_numeric_keys,
    convert_trace_graphs_to_pyg,
)

most_similar_trace_graphs = {i: trace_graphs[i] for i in most_similar_trace_graphs_gk}

# Modify node attribute value
# most_similar_trace_graphs["449315"]["process_execution"].nodes()["448967"]["attr"]["a"] -= 0.4

node_label_vocab, node_num_keys, edge_num_keys = build_vocab_and_numeric_keys(
    trace_graphs
)
data_most_similar_trace_graphs = convert_trace_graphs_to_pyg(
    most_similar_trace_graphs, node_label_vocab, node_num_keys, edge_num_keys
)

In [ ]:
import functools
import numpy as np  # numpy backend
import pygmtools as pygm
import networkx as nx  # for plotting graphs
import torch

# pygm.set_backend("numpy")  # set default backend for pygmtools
pygm.set_backend("pytorch")
np.random.seed(1)  # fix random seed

In [ ]:
graph_1_id = target_trace_graph_id
graph_2_id = list(most_similar_trace_graphs.keys())[0]

# attr_exclude_show = ["ocel:eid", "ocel:timestamp"]

G1 = trace_graphs[graph_1_id]["process_execution"]
G2 = trace_graphs[graph_2_id]["process_execution"]

n1 = len(G1.nodes)
n2 = len(G2.nodes)
A1 = torch.from_numpy(nx.to_numpy_array(G1))
A2 = torch.from_numpy(nx.to_numpy_array(G2))

conn1, edge1 = pygm.utils.dense_to_sparse(A1)
conn2, edge2 = pygm.utils.dense_to_sparse(A2)

gaussian_aff = functools.partial(
    pygm.utils.gaussian_aff_fn, sigma=0.1
)  # set affinity function
K = pygm.utils.build_aff_mat(
    None,
    edge1,
    conn1,
    None,
    edge2,
    conn2,
    None,
    None,
    None,
    None,
    # edge_aff_fn=gaussian_aff,
)

X = pygm.rrwm(
    K, n1, n2
)  # https://link.springer.com/chapter/10.1007/978-3-642-15555-0_36s
X = pygm.hungarian(X)

for i, g_i in enumerate(G1.nodes):
    g_j = list(G2.nodes)[np.argmax(X[i]).item()]

    # a_i = G1.nodes.data()[g_i]["attr"]
    # a_j = G2.nodes.data()[g_j]["attr"]
    # diff = {
    #     k: (a_i[k], a_j[k])
    #     for k in a_i
    #     if k in a_j and a_i[k] != a_j[k] and k not in attr_exclude_show
    # }
    # print(g_i, g_j, diff)

    l_i = G1.nodes.data()[g_i]["label"]
    l_j = G2.nodes.data()[g_j]["label"]
    if l_i != l_j:
        print(g_i, g_j, l_i, l_j)

In [ ]:
from scipy.optimize import linear_sum_assignment


def label_aware_graph_alignment_adj_matrix(
    G1, G2, label_weight=1.0, structure_weight=1.0
):
    """
    Aligns two graphs using Hungarian matching with label and adjacency pattern similarity.

    Parameters:
        G1, G2: networkx.Graph
            Graphs with 'label' attributes on nodes.
        label_weight: float
            Weight for label similarity score.
        structure_weight: float
            Weight for structural similarity score.

    Returns:
        mapping: dict
            Node mapping from G1 to G2.
        total_score: float
            Total similarity score for the alignment.
    """
    n1, n2 = len(G1.nodes), len(G2.nodes)
    if n1 != n2:
        raise ValueError(
            "Graphs must have the same number of nodes for this alignment method."
        )

    nodes1 = list(G1.nodes)
    nodes2 = list(G2.nodes)

    # Get adjacency matrices (order matters)
    A1 = nx.to_numpy_array(G1, nodelist=nodes1, dtype=int)
    A2 = nx.to_numpy_array(G2, nodelist=nodes2, dtype=int)

    # Create similarity matrix
    sim_matrix = np.zeros((n1, n2), dtype=float)

    for i, u in enumerate(nodes1):
        for j, v in enumerate(nodes2):
            score = 0.0

            # Label similarity
            if G1.nodes[u].get("label") == G2.nodes[v].get("label"):
                score += label_weight

            # Structural similarity: compare adjacency row patterns
            # (excluding self-loop position)
            row_u = np.delete(A1[i], i)
            row_v = np.delete(A2[j], j)

            # Similarity = number of matching adjacency entries
            structural_match_count = np.sum(row_u == row_v)
            score += structure_weight * structural_match_count

            sim_matrix[i, j] = score

    # Hungarian algorithm finds minimum cost, so negate similarity
    cost_matrix = -sim_matrix
    row_ind, col_ind = linear_sum_assignment(cost_matrix)

    mapping = {nodes1[i]: nodes2[j] for i, j in zip(row_ind, col_ind)}
    total_score = sim_matrix[row_ind, col_ind].sum()

    return mapping, total_score


mapping, score = label_aware_graph_alignment_adj_matrix(G1, G2)
print("Total Similarity Score:", score)

for g_i, g_j in mapping.items():
    # a_i = G1.nodes.data()[g_i]["attr"]
    # a_j = G2.nodes.data()[g_j]["attr"]
    # diff = {
    #     k: (a_i[k], a_j[k])
    #     for k in a_i
    #     if k in a_j and a_i[k] != a_j[k] and k not in attr_exclude_show
    # }
    # print(g_i, g_j, diff)

    l_i = G1.nodes.data()[g_i]["label"]
    l_j = G2.nodes.data()[g_j]["label"]
    if l_i != l_j:
        print(g_i, g_j, l_i, l_j)

In [ ]:
# Generate predictions for the most similar trace graphs using the loaded model
import torch
from torch_geometric.loader import DataLoader as PyGLoader

from gnn_graph_classification import GCNWithEdgeAgg

device = "cuda" if torch.cuda.is_available() else "cpu"

# Ensure `model` is available — if it's a state dict, rebuild a model skeleton
try:
    model
except NameError:
    model = torch.load(path.replace(".json", "-model_weights.pth"), weights_only=False)

# If a state-dict was saved instead of a model object, try to instantiate the known architecture
if (
    isinstance(model, dict)
    and "state_dict" not in model
    and not hasattr(model, "__call__")
):
    # attempt to import the model class from our gnn module
    try:
        in_ch = data_most_similar_trace_graphs[0].x.shape[1]
        model_obj = GCNWithEdgeAgg(in_ch)
        model_obj.load_state_dict(model)
        model = model_obj
    except Exception as e:
        raise RuntimeError(
            "Loaded object appears to be a state-dict but could not instantiate model: "
            + str(e)
        )

model = model.to(device)
model.eval()

# Use a non-shuffling loader so order matches the graph id list
pred_loader = PyGLoader(data_most_similar_trace_graphs, batch_size=8, shuffle=False)

preds = []
probs = []
with torch.no_grad():
    for batch in pred_loader:
        batch = batch.to(device)
        # Model forward for PyG-style GNNs: (x, edge_index, batch)
        out = model(batch.x, batch.edge_index, batch.batch)
        pprob = torch.nn.functional.softmax(out, dim=-1)
        preds.extend(out.argmax(dim=-1).cpu().numpy().tolist())
        probs.extend(pprob.cpu().numpy().tolist())

# Map predictions back to graph ids (data_most_similar_trace_graphs preserves insertion order)
graph_ids = list(most_similar_trace_graphs.keys())
results = []
for gid, pred, prob in zip(graph_ids, preds, probs):
    results.append(
        {"graph_id": gid, "pred": int(pred), "probabilities": [float(x) for x in prob]}
    )

# Expose results to the notebook
predictions = results

# Print a short summary
for r in predictions:
    print(r["graph_id"], "-> pred=", r["pred"], "prob=", r["probabilities"])

In [ ]:
nx.optimize_graph_edit_distance()